## 벡터스토어 실습

In [2]:
# LangSmith 추적 설정 부분
from dotenv import load_dotenv
import os

load_dotenv()

project_name = "wanted_rag_basic"
os.environ["LANGSMITH_PROJECT"] = project_name

In [3]:
from langchain_chroma import Chroma
from langchain.retrievers import EnsembleRetriever
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [4]:
file_path = "../../data/attention_article.pdf"

loader = PyPDFLoader(file_path)
docs = loader.load()

print(len(docs))
print(docs[0].page_content)


15
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. 

In [ ]:
# embedding 객체 생성

embedding = OpenAIEmbeddings(model="text-embedding-3-small")
len(embedding.embed_query("hello"))

1536

In [6]:
# PDF 파일을 쪼개기

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)

chunk_docs = splitter.split_documents(docs)

In [ ]:
# 벡터 스토어에 문서 저장

vectorstore = Chroma.from_documents(
    documents=chunk_docs,
    collection_name="attention_thesis",
    persist_directory="./chroma_store",
    embedding=embedding
)

In [ ]:
# 검색기 초기화

retriever = vectorstore.as_retriever(search_kwargs={"k":5})
retriever.invoke("Attention 의 개념이 뭐야?")

In [ ]:
# retriever.invoke("Attention 의 개념이 뭐야?")
# retriever.invoke("what is attention?")

[Document(id='5a234b6c-e579-495b-ac07-e47ffe677d45', metadata={'trapped': '/False', 'page_label': '13', 'moddate': '2024-04-10T21:11:43+00:00', 'creationdate': '2024-04-10T21:11:43+00:00', 'title': '', 'creator': 'LaTeX with hyperref', 'keywords': '', 'producer': 'pdfTeX-1.40.25', 'page': 12, 'source': '../../data/attention_article.pdf', 'author': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'total_pages': 15}, page_content='Attention Visualizations\nInput-Input Layer5\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<pad>\n<pad>\n<pad>\n<p

### 영어 번역 체인 만들기

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1-1. 번역용 프롬프트 템플릿
template = """ 
아래 들어오는 query를 영어로 번역해줘
##[Query]
{query}
"""

prompt_template = PromptTemplate.from_template(template)

# 1-2. LLM 초기화 (실제 ChatOpenAI 사용)
model = ChatOpenAI(
    temperature=0,
    model="gpt-4.1-mini",
    verbose=True
)

outputparser = StrOutputParser()

# 번역 체인: Prompt -> LLM -> OutputParser (출력: 영어 쿼리 문자열)
translation_chain = prompt_template | model | outputparser
print("--- [1단계] 한국어 -> 영어 번역 체인 구성 완료 ---")
translation_chain


--- [1단계] 한국어 -> 영어 번역 체인 구성 완료 ---


PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template=' \n아래 들어오는 query를 영어로 번역해줘\n##[Query]\n{query}\n')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000163F31332D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000163F314BDD0>, root_client=<openai.OpenAI object at 0x00000163F3149ED0>, root_async_client=<openai.AsyncOpenAI object at 0x00000163F314BB10>, model_name='gpt-4.1-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)
| StrOutputParser()

In [8]:
query = "Attention의 개념이 뭐야?"
translation_chain.invoke({"query":query})

'What is the concept of Attention?'

In [10]:
# 2. Retriever를 생성하고, 번역 체인 통합
vectorstore = Chroma(
    collection_name="attention_thesis",
    persist_directory="./chroma_store",
    embedding_function=embedding
)

# 로드된 vectorstore를 retriever로 변환
retriever = vectorstore.as_retriever(search_kwargs={"k":5})

print("--- [2-C단계] Retriever 구성 완료 ---")


retriever_chain = translation_chain | retriever
print("--- [3단계] 최종 통합 체인 구성 완료: 번역 -> 검색 ---")
retriever_chain

--- [2-C단계] Retriever 구성 완료 ---
--- [3단계] 최종 통합 체인 구성 완료: 번역 -> 검색 ---


PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template=' \n아래 들어오는 query를 영어로 번역해줘\n##[Query]\n{query}\n')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000163F31332D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000163F314BDD0>, root_client=<openai.OpenAI object at 0x00000163F3149ED0>, root_async_client=<openai.AsyncOpenAI object at 0x00000163F314BB10>, model_name='gpt-4.1-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)
| StrOutputParser()
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000163F3188750>, search_kwargs={'k': 5})

## 통합된 체인 실행 및 결과 확인

In [11]:
korean_query = "Attention 의 개념이 뭐야?"

try:
    # 이 invoke 시점에서 llm이 호출되어 쿼리를 번역하고, 그 결과로 Chroma가 검색됩니다.
    final_result = retriever_chain.invoke({"query":korean_query})

    print("\n--- [최종 결과] ---")
    print(f"체인 실행 완료. 최종 반환된 검색된 문서:")
    for i, doc in enumerate(final_result):
        print(f"[{i+1}] (Source: {doc.metadata.get('source', 'Unknown')}) 내용: {doc.page_content[:100]}...")

except Exception as e:
    print("Failed to Processing chain: {e}")


--- [최종 결과] ---
체인 실행 완료. 최종 반환된 검색된 문서:
[1] (Source: ../../data/attention_article.pdf) 내용: Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (r...
[2] (Source: ../../data/attention_article.pdf) 내용: Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what...
[3] (Source: ../../data/attention_article.pdf) 내용: described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism...
[4] (Source: ../../data/attention_article.pdf) 내용: Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governm...
[5] (Source: ../../data/attention_article.pdf) 내용: extremely small gradients 4. To counteract this effect, we scale the dot products by 1√dk
.
3.2.2 Mu...


## 3. RAG 체인 구성 (문서 포매팅 및 최종 응답)

In [ ]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

def format_docs(docs: List[Document]) -> str:
        return "\n\n".join([ 
        f"내용: {doc.page_content}\n출처: {doc.metadata.get('source', '정보 없음')}\n페이지: {doc.metadata.get('page_label', '정보 없음')}" 
        for doc in docs
    ])
# 최종 RAG 응답을 위한 한국어 프롬프트 템플릿
rag_template = """ 
당신은 주어진 문서(CONTEXT)를 이용하여 사용자의 질문(QUESTION)에 답변하는 유능한 챗봇입니다.
답변은 반드시 한국어로 작성해야 합니다. 응답에 사용한 문서의 출처와 페이지 정보를 응답 하단에 포함해주세요"

## [CONTEXT]
{context}

## [QUESTION]
{question}

## [ANSWER]
"""

final_rag_prompt = PromptTemplate.from_template(rag_template)

# 최종 RAG 체인 통합 (LCEL)

rag_chain = (
    RunnableParallel(
        {
            # 'context' 생성 경로: 원본 query를 받아서 번역 및 검색을 수행합니다.
            "context": (
                (lambda x: x["query"]) # 입력 딕셔너리에서 'query'값 (한국어 쿼리)를 추출
                | translation_chain
                | retriever_chain
                | format_docs
            ),
            # 'question'생성 경로: 원본 쿼리를 그대로 전달합니다.
            "question": (lambda x: x["query"]),
        }
    )
    | final_rag_prompt
    | model
    | outputparser
)

print("[4단계] 최종 RAG 체인 구성 완료: 흐름 질문 입력 -> 번역 -> 검색 -> 검색결과 포함해서 한국어 응답 생성")

[4단계] 최종 RAG 체인 구성 완료: 흐름 질문 입력 -> 번역 -> 검색 -> 검색결과 포함해서 한국어 응답 생성


In [18]:
# ------------------------
# 최종 결과 테스트
# ------------------------

answer = rag_chain.invoke({"query": "Attention에 대해서 설명해줘"})

print(answer)

Attention은 입력 시퀀스 내의 서로 다른 위치들 간의 관계를 계산하여 시퀀스의 표현을 만드는 메커니즘입니다. 특히, Self-attention(자기 주의)은 단일 시퀀스 내의 여러 위치들이 서로를 참조하여 정보를 통합하는 방식으로, 문장 내 단어들 간의 의존성을 효과적으로 포착할 수 있습니다. 

Scaled Dot-Product Attention은 Attention의 한 형태로, 쿼리(Query)와 키(Key)의 내적(dot product)을 계산한 후, 그 값을 키 차원의 제곱근으로 나누어 스케일링하고, 소프트맥스(softmax) 함수를 적용하여 각 값(Value)에 대한 가중치를 구합니다. 이 가중치들은 입력 값들에 곱해져 최종 출력이 생성됩니다. 수식으로는 다음과 같습니다:

Attention(Q, K, V) = softmax(QKᵀ / √d_k) V

여기서 Q는 쿼리 행렬, K는 키 행렬, V는 값 행렬이며, d_k는 키의 차원입니다.

또한, Multi-Head Attention은 여러 개의 Attention 레이어를 병렬로 실행하여 다양한 표현 공간에서 정보를 추출할 수 있게 합니다. 이를 통해 모델은 문장 내 다양한 관계를 동시에 학습할 수 있습니다.

Self-attention은 독립적인 RNN이나 CNN 없이도 입력과 출력을 효과적으로 표현할 수 있게 해주며, 읽기 이해, 요약, 문장 추론 등 다양한 자연어 처리 작업에서 성공적으로 활용되고 있습니다.

출처: ../../data/attention_article.pdf, 페이지 2, 4
